In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
from langchain import hub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.llms import Ollama

In [27]:
# 預處理documents

json_file_path = "example_documents.json"

with open(json_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# print(data["documents"][0]["content"])



In [ ]:


CHUNK_SIZE = 10
CHUNK_OVERLAP = 3

def process_documents(data, num_of_docs):
    documents = []
    original_content_dict = {}
    # 先處理每個文檔
    for doc in data["documents"]:
        original_content = doc["content"]
        
        # 創建基本的Document對象
        document = Document(
            page_content=original_content,
            metadata={
                "id": doc["id"],
                "title": doc["title"],
                "author": doc["author"],
                #"original_content": original_content  # 保存原始內容
            }
        )
        documents.append(document)
        original_content_dict[doc["id"]] = original_content
    
    # 創建text splitter並添加索引追踪
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # 分割文檔並添加索引信息
    splits = text_splitter.split_documents(documents)

    # 為每個分割添加索引信息
    # count_for_each_doc = {{}}

    for doc in splits:

        chunk_content = doc.page_content
        start_idx = original_content_dict[doc.metadata["id"]].find(chunk_content)
        end_idx = start_idx + len(chunk_content) - 1

        # if(doc.metadata["id"] not in count_for_each_doc):
        #     count_for_each_doc[doc.metadata["id"]] = 0
        
        # Stride=len(doc.page_content)
        # start_idx = count_for_each_doc[doc.metadata["id"]]

        # end_idx = start_idx + Stride -1
        # count_for_each_doc[doc.metadata["id"]] = end_idx + 1
        # 更新metadata，添加索引信息
        doc.metadata.update({
            "chunk_start_idx": start_idx,
            "chunk_end_idx": end_idx,
        })
        # 可以選擇刪除原始內容以節省空間
    
    return splits
# 0 1 2 3 (4) 5 6 7 (8) 9 10 11 (12) 13 14 15 (16) 17 18 19 (20)
# chunk_size = 5
# chunk_overlap = 1
# 處理文件

# 有幾個文檔，就有幾個count_for_each_doc

all_splits = process_documents(data, len(data["documents"]))

# 打印處理後的文件，包含索引信息
for split in all_splits:
    print(f"\n文檔ID: {split.metadata['id']}")
    print(f"標題: {split.metadata['title']}")
    print(f"內容: {split.page_content}")
    print(f"索引範圍: {split.metadata['chunk_start_idx']} 到 {split.metadata['chunk_end_idx']}")

In [30]:

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(all_splits, embeddings)


In [ ]:
# 建立QA系統
# See full prompt at https://smith.langchain.com/hub/langchain-ai/retrieval-qa-chat
llm = Ollama(model="llama3.1:8b")
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
rag_chain = create_retrieval_chain(vector_store.as_retriever(), combine_docs_chain)
# predined question
question = ["嬰兒發展研究有甚麼重要的發現?", "嬰兒發展研究有甚麼重要的發現?", "嬰兒發展研究有甚麼重要的發現?"]
final_result = []
for q in question:
    result = rag_chain.invoke({"input": q})
    final_result.append(result)


In [35]:
final_result_qa_string = ""
round_number = 1
for qa in final_result:
    # Document 對象需要使用 .metadata 來訪問元數據
    doc = qa['context'][0]  # 這是一個 Document 對象
    final_result_qa_string += (
        f"{round_number}. {qa['input']}\n"
        f"答:{qa['answer']}"
        f"根據[doc_id: {doc.metadata['id']}, "
        f"chunk_start_idx: {doc.metadata['chunk_start_idx']}, "
        f"chunk_end_idx: {doc.metadata['chunk_end_idx']}]\n"
    )
    round_number += 1

In [37]:
from langchain_core.prompts import PromptTemplate

one_shot_prompt = PromptTemplate.from_template(
    """
    有一系列的QA問題 每一個問題都有他對應的問題和答案，並且也有他這個問題所對應到的答案是根據哪個文檔的哪個段落來的，我希望你透過這些資訊，把這個QA問題及形成一份完整的內容，我的這個QA問題集會問出這個事件的脈絡和經過
    ，此外對於你寫出來的摘要對應到的每一段文字都要說明這段文字是來自哪個文檔的哪個段落

    以下一個列出QA問題和答案的範例(以美國大選為例)
    1. 這個事件中的主要人物有誰?
    答: 這個事件中有川普、賀錦麗、拜登、哈里斯(根據[doc_id: 1, chunk_start_idx: 10, chunk_end_idx: 20])
    2. 這個事件的時間軸是?
    答: 這個事件的時間軸是2024年11月5日(根據[doc_id: 2, chunk_start_idx: 30, chunk_end_idx: 40])
    3. 這個事件的經過是?
    答: 這個事件的經過是...(根據[doc_id: 3, chunk_start_idx: 50, chunk_end_idx: 60])
    4. 這個事件的結果是?
    答: 這個事件的結果是...(根據[doc_id: 4, chunk_start_idx: 70, chunk_end_idx: 80])
    5. 這個事件的影響是?
    答: 這個事件的影響是...(根據[doc_id: 5, chunk_start_idx: 90, chunk_end_idx: 100])
    6. 這個事件的意義是?
    答: 這個事件的意義是...(根據[doc_id: 6, chunk_start_idx: 30, chunk_end_idx: 120])

    摘要範例
    在2024年的美國大選中，川普、賀錦麗、拜登與哈里斯成為這場政治較量中的核心人物（根據[doc_id: 1, chunk_start_idx: 10, chunk_end_idx: 20]）。這次選舉於2024年11月5日正式展開，這一天標誌了美國政治舞台上的重要轉折點，無數選民的選擇集中在這一刻，訴說著美國選舉體制的意涵（根據[doc_id: 2, chunk_start_idx: 30, chunk_end_idx: 40]）。
    在競選過程中，川普與拜登多次激烈辯論，展示了彼此的政策願景及施政方針。川普以一貫強硬的立場面對公共政策問題，而拜登則強調社會團結和環保議題。另一方面，賀錦麗與哈里斯分別代表兩黨輔佐競選，致力於拉攏支持者，讓民眾能夠更加明確地理解兩黨的政策異同（根據[doc_id: 3, chunk_start_idx: 50, chunk_end_idx: 60]）。在這樣的激烈交鋒中，美國選民的支持逐漸傾向於拜登，最終，他在選舉中勝出，成功成為美國總統，而川普未能再度連任。這一結果引起了各地選民的激烈反應，許多地方爆發了不同政見的抗議活動，表現出民眾對此次選舉結果的複雜情緒（根據[doc_id: 4, chunk_start_idx: 70, chunk_end_idx: 80]）。
    隨著拜登的當選，美國的內政外交迎來了新氣象，他在上任後推行了一系列環保和經濟改革，逐步改變了美國的國內外政策格局，這些舉措也帶來了對未來美國發展方向的深刻影響（根據[doc_id: 5, chunk_start_idx: 90, chunk_end_idx: 100]）。而這次大選的意義並不僅限於政權更替，更是一場對於美國民主體制的深刻檢視。它顯示了美國選民在當代政治中愈發關鍵的角色，並且凸顯了兩黨對立的現實，激起了關於政治分歧與社會團結的討論，這些討論將在未來繼續發酵，對美國的民主制度和社會發展產生深遠的影響（根據[doc_id: 6, chunk_start_idx: 30, chunk_end_idx: 120]）。

    現在請你根據這個範例，把我的QA問題和答案轉換成一份完整的內容

    我的QA問題和答案
    {input}



    """
)

answer_to_abstract_chain = one_shot_prompt | llm
result = answer_to_abstract_chain.invoke({"input": final_result_qa_string})
print(result)

In [ ]:
prompt = PromptTemplate.from_template(
    """
你是一位專業的文章撰寫者，請你根據以下的問答內容，撰寫一份完整且連貫的摘要報告。

問答內容：
{input}

撰寫要求：
1. 內容結構：
   - 按照時間順序或邏輯順序組織內容
   - 確保段落之間的轉折自然
   - 使用適當的連接詞來增加文章流暢度

2. 引用格式：
   - 每個論述都必須標註來源：（根據[doc_id: X, chunk_start_idx: Y, chunk_end_idx: Z]）
   - 來源標註要放在相關敘述的結尾處
   - 如果一個段落包含多個來源的內容，需分別標註

3. 寫作風格：
   - 使用客觀、專業的語氣
   - 避免重複QA中的問題形式
   - 將問答轉化為敘事性的描述

4. 內容完整性：
   - 確保涵蓋所有QA中的重要信息
   - 適當整合相關信息，避免過度分散
   - 在不同觀點間取得平衡

請根據以上要求，將QA內容改寫成一份連貫的摘要報告。報告應該能讓讀者清楚理解事件的來龍去脈，同時保持專業性和可信度。

輸出格式示例：
[完整的敘事性摘要，每個論述都需包含來源標註]

注意：請確保每個重要信息都有對應的文檔來源標註，且內容的組織要合乎邏輯，讓讀者能夠輕鬆理解整個事件的發展脈絡。
"""
)

answer_to_abstract_chain = prompt | llm
result = answer_to_abstract_chain.invoke({"input": final_result_qa_string})

print(result)